# Adding active learning to an existing River application

If you already have a River application, `fl.active.Wrapper` lets you add output-level active
learning to it with a few changed lines. It mirrors `river.active`: `predict_one` returns the
prediction along with what to ask for. The difference is that it returns a *set of outputs*
instead of a boolean, and that queries are paid for with a budget.

We use River's `SolarFlare` dataset, which has three outputs: the number of C, M and X class
solar flares in 24 hours.

In [1]:
from river import datasets, linear_model, metrics, multioutput, preprocessing

import fishladder as fl

dataset = datasets.SolarFlare()
OUTPUTS = ["c-class-flares", "m-class-flares", "x-class-flares"]
x, y = next(iter(dataset))
x, y

({'zurich-class': 'H',
  'largest-spot-size': 'A',
  'spot-distribution': 'X',
  'activity': 1,
  'evolution': 3,
  'previous-24h-flare-activity': 1,
  'hist-complex': 1,
  'hist-complex-this-pass': 1,
  'area': 1,
  'largest-spot-area': 1},
 {'c-class-flares': 0, 'm-class-flares': 0, 'x-class-flares': 0})

## A typical River application

All the features are categorical, so the pipeline one-hot encodes them before a multi-output
regressor. In a typical River loop, the model learns from every label.

In [2]:
def make_model():
    return preprocessing.OneHotEncoder() | multioutput.PerOutputRegressor(
        linear_model.LinearRegression()
    )


model = make_model()
metric = metrics.multioutput.PerOutput(metrics.MAE())

for x, y in dataset:
    y_pred = model.predict_one(x)
    if y_pred:
        metric.update(y, y_pred)
    model.learn_one(x, y)

metric

c-class-flares - MAE: 0.463365
m-class-flares - MAE: 0.111064
x-class-flares - MAE: 0.014723

## The same application with active learning

Suppose a C-class count is cheap to obtain while M and X classes need an expert. We wrap the same
pipeline, and only learn from the outputs the wrapper asks for. The changes are the wrapper and the
two lines that ask for labels.

In [3]:
COSTS = {"c-class-flares": 1.0, "m-class-flares": 3.0, "x-class-flares": 3.0}

model = fl.active.Wrapper(
    model=make_model(),
    strategy=fl.strategy.RandomPerOutput(p=0.5, seed=42),
    budget=fl.budget.RateBudget(1.0),
    costs=COSTS,
)
metric = metrics.multioutput.PerOutput(metrics.MAE())
asked = {o: 0 for o in OUTPUTS}

for x, y in dataset:
    y_pred, ask = model.predict_one(x)
    # Outputs that were never queried yet have no prediction
    if all(o in y_pred for o in y):
        metric.update(y, y_pred)
    if ask:
        model.learn_one(x, {o: y[o] for o in ask})
        for o in ask:
            asked[o] += 1

print(f"spent {model.budget.spent:.0f} over {model.budget.n_seen} instances")
print(asked)
metric

spent 1065 over 1066 instances
{'c-class-flares': 423, 'm-class-flares': 107, 'x-class-flares': 107}


c-class-flares - MAE: 0.468004
m-class-flares - MAE: 0.076473
x-class-flares - MAE: 0.005671

`RateBudget(1.0)` makes one cost unit available per instance, against 7 for querying everything,
and the expensive outputs get fewer queries. In a real application, `{o: y[o] for o in ask}` is
where you would request the labels from an annotator or a lab.

A word of caution on the numbers: most flare counts are zero, so on this dataset a model that
predicts values close to zero already gets a low MAE. The errors of the two loops are not a
verdict on active learning. This notebook is about the mechanics. Each call to `predict_one` is a query decision: the cost of what it asks for
is charged to the budget right away.

## From the application to an experiment

The loop above is what `fl.evaluate.prequential` runs for you, plus a full log of what happened.
Since the evaluator is built on the same wrapper, what you measure is what you deploy.

In [4]:
log = fl.evaluate.prequential(
    dataset,
    make_model(),
    fl.strategy.RandomPerOutput(p=0.5, seed=42),
    fl.budget.RateBudget(1.0),
    COSTS,
)

df = log.to_frame()
df.groupby("output").agg(queries=("queried", "sum"), cost=("cost", "sum"))

,queries,cost
output,,
c-class-flares,423,423.0
m-class-flares,107,321.0
x-class-flares,107,321.0


In [5]:
fl.metrics.per_output(log, metrics.MAE())

{'c-class-flares': MAE: 0.465365,
 'm-class-flares': MAE: 0.076185,
 'x-class-flares': MAE: 0.005671}